# Capítulo 8 — Filas: quantas equipes na doca

## Missão
Caminhões chegam à doca de carga, em média 9 por hora, e uma equipe descarrega 12 por hora. Caminhão parado custa R$ 180/h e cada equipe, R$ 95/h. Você vai calcular as medidas do **M/M/1** e do **M/M/c** com as fórmulas do livro, confirmar a **lei de Little**, escolher o número de equipes que minimiza o custo total e validar tudo com uma **simulação de eventos**.

**No fim deste caderno você terá:** o módulo `gabarito_otimizador/filas.py` (M/M/1, M/M/c com Erlang C e um simulador de fila) e a decisão de quantas equipes contratar.

**Leitura essencial (opcional, livro em inglês):**
- 8.1.1 Elements of the Queuing System · PDF p. 265
- 8.1.2 Kendall Notation · PDF p. 267
- 8.2 Single-Server Queuing Model with Poisson Arrivals and Exponential Service Times · PDF p. 268
- 8.3 Multiple-Server Queuing Model · PDF p. 274
- 8.4 Little's Relationships for Queuing Models · PDF p. 281

**Gabarito:** a mesma trilha com o código pronto. Tente primeiro em `praticar/`.

## Preparação
Configuração do caderno (já vem pronta). Rode antes de tudo.

In [ ]:
import os, sys, json, time, math, random
sys.path.insert(0, "../comum")                 # dados.py (dados fictícios do projeto)
sys.path.insert(0, "..")                       # raiz: onde fica o seu pacote gabarito_otimizador/
os.makedirs("../gabarito_otimizador", exist_ok=True)
open("../gabarito_otimizador/__init__.py", "a").close()        # gabarito_otimizador/ é o projeto que você constrói ao longo do livro
os.makedirs("../saidas", exist_ok=True)        # arquivos gerados nos exercícios
from dados import *
import warnings; warnings.filterwarnings("ignore")
%load_ext autoreload
%autoreload 2
print("Pronto. Python", sys.version.split()[0], "| seu pacote: gabarito_otimizador")

## Ideia em 30 segundos
Na notação de Kendall M/M/c, as chegadas são de Poisson (taxa λ), o atendimento é exponencial (taxa μ) e há c servidores. A utilização é `ρ = λ/(cμ)` e precisa ser menor que 1.

**No M/M/1:**
- `L = ρ/(1-ρ)` caminhões no sistema;
- `W = 1/(μ-λ)` horas no sistema;
- `Lq = ρ²/(1-ρ)` caminhões na fila;
- `Wq = ρ/(μ-λ)` horas na fila.

**Lei de Little:** `L = λ·W` vale para quase qualquer fila.

### 8.1 **O módulo de filas.** Crie `filas.py` com:
- `mm1(lam, mu)`: um dicionário com ρ, L, Lq, W, Wq e `P0`;
- `mmc(lam, mu, c)`: com a fórmula de Erlang C para a probabilidade de esperar e `Lq = P(esperar)·ρ/(1-ρ)`; o resto pela lei de Little;
- `simular(lam, mu, c, n_clientes, semente)`: sorteia as chegadas e os atendimentos exponenciais e devolve as médias de Wq e de W.

_A célula grava `gabarito_otimizador/filas.py`: a 1ª linha (`%%writefile`) já vem pronta, digite o código embaixo dela._

In [ ]:
%%writefile ../gabarito_otimizador/filas.py
"""Teoria das filas: fórmulas M/M/1 e M/M/c e um simulador simples."""
import heapq
from math import factorial

import numpy as np


def mm1(lam, mu):
    rho = lam / mu
    if rho >= 1:
        raise ValueError("fila instável: λ >= μ")
    return {"rho": rho, "P0": 1 - rho, "L": rho / (1 - rho), "Lq": rho ** 2 / (1 - rho),
            "W": 1 / (mu - lam), "Wq": rho / (mu - lam)}


def mmc(lam, mu, c):
    a, rho = lam / mu, lam / (c * mu)
    if rho >= 1:
        raise ValueError("fila instável: λ >= cμ")
    soma = sum(a ** k / factorial(k) for k in range(c))
    ultimo = a ** c / (factorial(c) * (1 - rho))
    p0 = 1 / (soma + ultimo)
    p_espera = ultimo * p0
    Lq = p_espera * rho / (1 - rho)
    Wq = Lq / lam
    return {"rho": rho, "P0": p0, "P_esperar": p_espera, "Lq": Lq, "Wq": Wq, "W": Wq + 1 / mu, "L": lam * (Wq + 1 / mu)}


def simular(lam, mu, c, n_clientes=50000, semente=0):
    r = np.random.default_rng(semente)
    chegadas = np.cumsum(r.exponential(1 / lam, n_clientes))
    servico = r.exponential(1 / mu, n_clientes)
    livres = [0.0] * c
    heapq.heapify(livres)
    esperas, sistema = np.empty(n_clientes), np.empty(n_clientes)
    for i, (t, s) in enumerate(zip(chegadas, servico)):
        livre = heapq.heappop(livres)
        inicio = max(t, livre)
        heapq.heappush(livres, inicio + s)
        esperas[i], sistema[i] = inicio - t, inicio + s - t
    return {"Wq": float(esperas.mean()), "W": float(sistema.mean())}

### 8.2 **M/M/1 com uma equipe.** Calcule as medidas com `CHEGADAS_HORA` e `ATENDIMENTO_HORA`, mostre as esperas em **minutos** e confira a lei de Little (`L = λW` e `Lq = λWq`).

In [ ]:
from gabarito_otimizador.filas import mm1, mmc, simular
lam, mu = CHEGADAS_HORA, ATENDIMENTO_HORA
m = mm1(lam, mu)
print({k: round(v, 3) for k, v in m.items()})
print(f"espera na fila {m['Wq'] * 60:.1f} min | no sistema {m['W'] * 60:.1f} min")
print("Little:", round(m["L"], 4), "=", round(lam * m["W"], 4), "|", round(m["Lq"], 4), "=", round(lam * m["Wq"], 4))

### 8.3 **Simulação confirma.** Rode o seu simulador com 1 equipe e compare Wq e W com as fórmulas. Depois desenhe como Wq explode quando ρ se aproxima de 1 (λ de 1 a 11,5).

In [ ]:
import matplotlib.pyplot as plt
s = simular(lam, mu, 1)
print("simulação Wq:", round(s["Wq"] * 60, 1), "min | fórmula:", round(m["Wq"] * 60, 1), "min")
lams = np.linspace(1, 11.5, 50)
plt.plot(lams / mu, [mm1(l, mu)["Wq"] * 60 for l in lams]); plt.xlabel("ρ = λ/μ"); plt.ylabel("espera na fila (min)")
plt.title("A fila explode perto de ρ = 1"); plt.show()

### 8.4 **Mais equipes: M/M/c.** Para c = 1 a 4, mostre ρ, a chance de esperar, Wq em minutos e o **custo total por hora** = `CUSTO_EQUIPE·c + CUSTO_CAMINHAO_PARADO·L`. Quantas equipes minimizam o custo? Confira o c ótimo com a simulação.

In [ ]:
linhas = []
for c in range(1, 5):
    r_ = mmc(lam, mu, c)
    linhas.append({"equipes": c, "rho": r_["rho"], "P_esperar": r_["P_esperar"], "Wq_min": r_["Wq"] * 60,
                   "custo_h": CUSTO_EQUIPE * c + CUSTO_CAMINHAO_PARADO * r_["L"]})
t = pd.DataFrame(linhas).set_index("equipes").round(3); display(t)
c_ot = int(t["custo_h"].idxmin())
print("melhor:", c_ot, "equipes | simulação Wq:", round(simular(lam, mu, c_ot)["Wq"] * 60, 2), "min | fórmula:", round(t.loc[c_ot, "Wq_min"], 2))

### 8.5 **Uma equipe rápida ou duas lentas?** Compare um M/M/1 com μ = 24 contra um M/M/2 com μ = 12 cada (a mesma capacidade total). Qual dá a menor espera na fila e qual dá o menor tempo no sistema? Explique num comentário.

In [ ]:
um_rapido, dois_lentos = mm1(lam, 24), mmc(lam, 12, 2)
print(f"1 rápida: Wq {um_rapido['Wq'] * 60:.2f} min | W {um_rapido['W'] * 60:.2f} min")
print(f"2 lentas: Wq {dois_lentos['Wq'] * 60:.2f} min | W {dois_lentos['W'] * 60:.2f} min")
# Duas equipes esperam menos na fila (raramente as duas estão ocupadas), mas cada descarga leva o dobro do tempo:
# no sistema, a equipe única e rápida ganha.

## Fechamento

- O que eu construí hoje:
- O que me surpreendeu:
- O que quero tentar sozinho a seguir:
